In [43]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling
)

texts = [
    "Artificial intelligence enables machines to perform tasks that normally require human intelligence.",
    "Machine learning is a branch of artificial intelligence that learns patterns from data.",
    "Deep learning uses neural networks with multiple layers to learn complex representations.",
    "Natural language processing allows computers to understand and generate human language.",
    "Generative AI creates new content such as text, images, music, and computer code.",
    "Large language models are trained on large amounts of text to predict the next token.",
    "Transformers use attention mechanisms to learn relationships between words and tokens.",
    "GPT models generate text by predicting one token at a time based on previous tokens.",
    "Fine-tuning adapts a pretrained language model to a specific task or domain.",
    "Tokenization converts text into tokens and maps them to numerical token IDs.",
    "Embedding layers convert token IDs into dense numerical vectors.",
    "Training a language model involves minimizing the difference between predicted and actual next tokens.",
    "Artificial intelligence can support healthcare, education, robotics, and scientific research.",
    "Responsible AI requires attention to fairness, privacy, safety, and reliability.",
    "Data quality plays an important role in the performance of machine learning models.",
    "A neural network learns useful patterns by updating its parameters during training.",
    "Transfer learning reuses knowledge learned by a pretrained model.",
    "Text generation quality depends on training data, model parameters, and decoding strategies."
]

data = Dataset.from_dict({"text": texts})

In [44]:
model_name = "distilgpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.pad_token_id

Loading weights: 100%|██████████| 76/76 [00:00<00:00, 7201.01it/s]


In [45]:
def tokenize(example):
    return tokenizer(
        example['text'],
        truncation=True,
        max_length=128
    )

tokenized_dataset = data.map(
    tokenize,
    batched=True,
    remove_columns=['text']
)

Map: 100%|██████████| 18/18 [00:00<00:00, 831.98 examples/s]


In [46]:
splited_dataset = tokenized_dataset.train_test_split(
    test_size=0.2,
    seed=42
)

train_dataset = splited_dataset["train"]
test_dataset = splited_dataset["test"]

data_collator = DataCollatorForLanguageModeling(
    mlm=False,
    tokenizer=tokenizer
)

In [47]:
training_args = TrainingArguments(
    num_train_epochs=5,
    learning_rate=5e-5,
    weight_decay=0.01,
    logging_strategy="epoch",
)

trainer = Trainer(
    model = model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    data_collator=data_collator
)

trainer.train()

Step,Training Loss
2,4.611441
4,3.945742
6,3.728763
8,3.487230
10,3.345422


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.84it/s]


TrainOutput(global_step=10, training_loss=3.823719692230225, metrics={'train_runtime': 4.6085, 'train_samples_per_second': 15.189, 'train_steps_per_second': 2.17, 'total_flos': 319986450432.0, 'train_loss': 3.823719692230225, 'epoch': 5.0})

In [48]:
results = trainer.evaluate()

eval_loss = results["eval_loss"]
perplexity = np.exp(eval_loss)
perplexity

/Users/aryanahuja/Documents/Aryan/GAI ESE Prac/env/lib/python3.11/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Step
3.345422,4.187350,10


np.float64(65.84804876868141)